## Import libraries and load Gemini API key


In [ ]:
!pip install -q -U google-generativeai

In [ ]:
# Import necessary libraries
from google import genai
from google.genai import types
from google.colab import userdata
import pandas as pd
import re
from tqdm import tqdm
tqdm.pandas()
import os

# Load Gemini API key from Colab secure variables
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
# Initialize Gemini client
client = genai.Client(api_key=GEMINI_API_KEY)

# Select model
# MODEL = "gemini-3.1-pro-preview"
MODEL = "gemini-3-flash-preview"

# Each model has a daily usage limit
if MODEL == "gemini-3.1-pro-preview":
    # Daily API limit (we leave a margin of 10 over the actual limit of 250)
    RPD_LIMIT = 240
else:
    RPD_LIMIT = 1000

## Load dataset


In [2]:
# connect Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Load individual datasets per task
path = "/content/drive/MyDrive/tfg/corpusMentalRiskES/splits/LLM/"
path_tara5 = path + "tara5/"

df_anx = pd.read_csv(path + "anxiety_llm.csv", sep=",")
df_dep = pd.read_csv(path + "depression_llm.csv", sep=",")
df_ed = pd.read_csv(path + "ed_llm.csv", sep=",")
df_multi = pd.read_csv(path + "multiclass_llm.csv", sep=",")

df_anx_tara5 = pd.read_csv(path_tara5 + "anxiety_llm_tara5.csv")
df_dep_tara5 = pd.read_csv(path_tara5 + "depression_llm_tara5.csv")
df_ed_tara5  = pd.read_csv(path_tara5 + "ed_llm_tara5.csv")
df_multi_tara5 = pd.read_csv(path_tara5 + "multiclass_llm_tara5.csv")

# Select classification task: anx | dep | ed | multi
task = "anx" # <--- Change here

# Assign the dataframe corresponding to the selected task
if task == "anx":
  df = df_anx
  df_tara5 = df_anx_tara5
elif task == "dep":
  df = df_dep
  df_tara5 = df_dep_tara5
elif task == "ed":
  df = df_ed
  df_tara5 = df_ed_tara5
elif task == "multi":
  df = df_multi
  df_tara5 = df_multi_tara5

## Function to build the prompt


The text to be classified is added to a fixed prompt that changes according to the task.

> Note: Since the study focuses on Spanish texts, the prompts passed to the models remain in Spanish.


In [4]:
def build_prompt(text, task):
  """ Dynamic construction of the prompt according to the task.
  Returns the complete text to be sent to the model.
  """

  if task == "anx":
    return f"""
Vas a recibir un texto que corresponde a una serie de mensajes escritos por una persona en una red social.
El texto puede o no reflejar indicios de ansiedad.

La ansiedad se asocia a una preocupación persistente y difícil de controlar, acompañada de nerviosismo, inquietud interna y una sensación continua de amenaza o anticipación negativa. Estas experiencias suelen generar malestar emocional sostenido y dificultad para relajarse.

Tu tarea es clasificar el texto únicamente en una de las siguientes categorías:

1: el texto muestra indicios de ansiedad
0: el texto no muestra indicios de ansiedad

El contenido de los mensajes se proporciona a continuación entre comillas triples:

\"\"\"{text}\"\"\"

Responde solo con el número correspondiente.
"""
  elif task == "dep":
    return f"""
Vas a recibir un texto que corresponde a una serie de mensajes escritos por una persona en una red social.
El texto puede o no reflejar indicios de depresión.

La depresión se asocia a un estado persistente de tristeza, apatía o vacío emocional, acompañado de una disminución del interés o la capacidad de disfrute en actividades habituales. Estas experiencias pueden incluir pensamientos negativos recurrentes, fatiga emocional y una percepción pesimista de uno mismo o del futuro.

Tu tarea es clasificar el texto únicamente en una de las siguientes categorías:

1: el texto muestra indicios de depresión
0: el texto no muestra indicios de depresión

El contenido de los mensajes se proporciona a continuación entre comillas triples:

\"\"\"{text}\"\"\"

Responde solo con el número correspondiente.
"""
  elif task == "ed":
    return f"""
Vas a recibir un texto que corresponde a una serie de mensajes escritos por una persona en una red social.
El texto puede o no reflejar indicios de un trastorno de la conducta alimentaria.

Los trastornos de la conducta alimentaria se asocian a una preocupación persistente por el peso corporal, la alimentación o la imagen corporal, que puede manifestarse mediante pensamientos rígidos o negativos sobre la comida, el cuerpo o el control del peso. Estas preocupaciones suelen generar malestar emocional, culpa, ansiedad en torno a la alimentación y conductas de evitación o control.

Tu tarea es clasificar el texto únicamente en una de las siguientes categorías:

1: el texto muestra indicios de un trastorno de la conducta alimentaria
0: el texto no muestra indicios de un trastorno de la conducta alimentaria

El contenido de los mensajes se proporciona a continuación entre comillas triples:

\"\"\"{text}\"\"\"

Responde solo con el número correspondiente.
"""
  else:
    return f"""
Vas a recibir un texto que corresponde a una serie de mensajes escritos por una persona en una red social.
El texto puede reflejar indicios de uno de los siguientes estados psicológicos, o no reflejar ninguno de ellos.

Ansiedad: se asocia a una preocupación persistente y difícil de controlar, acompañada de nerviosismo, inquietud interna y una sensación continua de amenaza o anticipación negativa, generando malestar emocional y dificultad para relajarse.

Depresión: se asocia a un estado persistente de tristeza, apatía o vacío emocional, junto con una disminución del interés o la capacidad de disfrute en actividades habituales, pensamientos negativos recurrentes y una visión pesimista de uno mismo o del futuro.

Trastornos de la conducta alimentaria: se asocian a una preocupación persistente por el peso, la alimentación o la imagen corporal, que puede manifestarse mediante pensamientos rígidos o negativos sobre la comida o el cuerpo, así como malestar emocional, culpa o ansiedad en torno a la alimentación.

Tu tarea es clasificar el texto únicamente en una de las siguientes categorías:

0: el texto no muestra indicios de ninguno de los estados descritos
1: el texto muestra indicios de ansiedad
2: el texto muestra indicios de depresión
3: el texto muestra indicios de un trastorno de la conducta alimentaria

El contenido de los mensajes se proporciona a continuación entre comillas triples:

\"\"\"{text}\"\"\"

Responde solo con el número correspondiente.
"""

## Calculate the number of tokens needed to process the dataset

In [ ]:
def total_tokens_dataset(df, task):
  """ Calculates the total number of input tokens
  for a complete dataset, using the model's tokenizer
  """
  total_tokens = 0

  for text in df["texto"]:
    # Count tokens of the complete prompt (instructions + text)
    response = client.models.count_tokens(
        model=MODEL,
        contents=build_prompt(text, task),
    )
    total_tokens += response.total_tokens

  return total_tokens


# Token calculation per dataset
total_anx = total_tokens_dataset(df_anx, "anx")
print("ANXIETY DATASET")
print("Tokens:", total_anx)
print("Average per text:", total_anx / len(df_anx))

total_dep = total_tokens_dataset(df_dep, "dep")
print("DEPRESSION DATASET")
print("Tokens:", total_dep)
print("Average per text:", total_dep / len(df_dep))

total_ed = total_tokens_dataset(df_ed, "ed")
print("ED DATASET")
print("Tokens:", total_ed)
print("Average per text:", total_ed / len(df_ed))

total_multi = total_tokens_dataset(df_multi, "multi")
print("MULTICLASS DATASET")
print("Tokens:", total_multi)
print("Average per text:", total_multi / len(df_multi))

# Total sum of tokens needed for the entire experiment
print(f"Total tokens: {total_anx + total_dep + total_ed + total_multi}")

DATASET ANSIEDAD
Tokens: 445813
Media por texto: 891.626
DATASET DEPRESIÓN
Tokens: 378367
Media por texto: 758.2505010020041
DATASET ED
Tokens: 260571
Media por texto: 777.823880597015
DATASET MULTICLASE
Tokens: 1273296
Media por texto: 954.4947526236881
Total de tokens: 2358047


## Zero-shot classification with Gemini


In [ ]:
def classify_text(text, task):
  """ Classification function using Gemini
  Returns:
  - int (0,1,2,3) if there is a valid prediction
  - None if the model blocks or does not return text
  """

  response = client.models.generate_content(
    model=MODEL,
    contents=build_prompt(text, task),
    config={
        "temperature": 0,       # Deterministic generation
        # Force the model to respond directly without spending tokens on internal reasoning
        "thinking_config": {
            "thinking_level": "MINIMAL"
        }
    }
  )

  # Returns None if there is no text
  # (usually happens because the prompt is blocked by PROHIBITED_CONTENT)
  if not response.text:
    return None

  # Extract class number using regular expression
  match = re.search(r"[0-3]", response.text)

  return int(match.group()) if match else None

Note: The Gemini 3.1 Pro API (preview) has a limit of 250 RPD (requests per day) that is not increased even with billing enabled, as it is a preview model. Therefore, classification is performed in daily sessions of up to RPD_LIMIT calls, saving a checkpoint in each execution to resume the next day without losing progress.


In [ ]:
import time

checkpoint_dir = "/content/drive/MyDrive/tfg/corpusMentalRiskES/checkpoints/"
output_dir = "/content/drive/MyDrive/tfg/corpusMentalRiskES/resultados/"

os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

def classify_with_daily_limit(df, task, rpd_limit=RPD_LIMIT, run_id=None):
  """
  Classifies the dataset respecting the daily API limit.

  Logic:
  - Loads the checkpoint if it exists (resumes from where it left off)
  - Processes up to rpd_limit texts per execution
  - Saves the checkpoint at the end of the day's session
  - When all texts are classified, saves the final result

  Parameters:
  - df: DataFrame with 'texto' column
  - task: task ('anx', 'dep', 'ed', 'multi')
  - rpd_limit: maximum API calls in this execution
  """

  if run_id is None:
    checkpoint_path = checkpoint_dir + f"checkpoint_{MODEL.replace('/', '_')}_{task}.csv"
  else:
    checkpoint_path = checkpoint_dir + f"tara5_{MODEL.replace('/', '_')}_{task}_run{run_id}.csv"

  # --- Load checkpoint if it exists ---
  if os.path.exists(checkpoint_path):
    df_ckpt = pd.read_csv(checkpoint_path)
    df["pred"] = df_ckpt["pred"]
    already_processed = int(df["pred"].notna().sum())
    print(f"✅ Checkpoint found: {already_processed}/{len(df)} texts already processed.")
  else:
    df["pred"] = None
    already_processed = 0
    print(f"🆕 No previous checkpoint. Starting from scratch.")

  pending = int(df["pred"].isna().sum())

  if pending == 0:
    print("🎉 Dataset already completed. Nothing to process.")
    return df

  calls_today = 0
  blocked = 0

  print(f"📋 Pending texts: {pending} | Today's limit: {rpd_limit}")
  print(f"📅 Estimated remaining days: {-(-pending // rpd_limit)}")  # ceil division

  # Progress bar on the texts we will process today
  pbar = tqdm(total=min(pending, rpd_limit), desc="Today's progress")

  for i, row in df.iterrows():

    # Stop if we reach the daily limit
    if calls_today >= rpd_limit:
      print(f"\n🛑 Daily limit reached ({rpd_limit} calls). Come back tomorrow.")
      break

    # Skip already processed rows
    if pd.notna(df.at[i, "pred"]):
      continue

    start_time = time.time()

    df.at[i, "pred"] = classify_text(row["texto"], task)
    calls_today += 1

    if df.at[i, "pred"] is None:
      blocked += 1

    pbar.update(1)
    pbar.set_postfix({"blocked": blocked, "hoy": calls_today})

    # Rate limit control (25 RPM)
    elapsed = time.time() - start_time
    sleep_time = max(0, 2.5 - elapsed)
    time.sleep(sleep_time)

  pbar.close()

  # --- Save checkpoint ---
  df.to_csv(checkpoint_path, index=False)
  total_processed = int(df["pred"].notna().sum())
  print(f"\n💾 Checkpoint saved. Total processed: {total_processed}/{len(df)}")

  # --- If dataset is complete, save final result ---
  if df["pred"].isna().sum() == 0:
    output_path = output_dir + f"{MODEL.replace('/', '_')}_{task}.csv"
    df.to_csv(output_path, index=False)
    print(f"🎉 Dataset completed. Final result saved in:\n   {output_path}")

  return df, calls_today

In [ ]:
# Apply classification to the selected dataset
# The prediction is stored in a new column "pred"
print(f"Starting session | Task: {task} | Model: {MODEL}")
df, _ = classify_with_daily_limit(df, task)

In [ ]:
df.to_csv(checkpoint_dir + f"checkpoint_{MODEL.replace('/', '_')}_{task}.csv", index=False)
print("Checkpoint saved manually")

Checkpoint guardado manualmente


### Predictions check


In [ ]:
from sklearn.metrics import f1_score, accuracy_score

# Replace None with an invalid class so it counts as an error
pred_labels = df["pred"].fillna(-1).astype(int)
mask = pred_labels != -1  # Exclude blocks

# If it is binary classification
if task in ["anx", "dep", "ed"]:
    # Column with the actual label
    true_labels = df["bs"]
    f1 = f1_score(true_labels[mask], pred_labels[mask], average="binary")
else:
    # Multiclass classification
    true_labels = df["label_mc"]
    f1 = f1_score(true_labels[mask], pred_labels[mask], average="macro")

acc = accuracy_score(true_labels[mask], pred_labels[mask])

print("F1-score:", round(f1, 4))
print("Acc:", round(acc, 4))

F1-score: 0.9581
Acc: 0.9257


In [ ]:
# Number of blocks
num_blocked = df["pred"].isna().sum()
block_rate = num_blocked / len(df)

print("Blocks:", num_blocked)
print("Block rate:", round(block_rate, 4))

Bloqueos: 2
Tasa de bloqueo: 0.004


### Store predictions


In [ ]:
output_path = (
    f"/content/drive/MyDrive/tfg/corpusMentalRiskES/resultados/"
    f"{MODEL}_{task}.csv"
)
df.to_csv(output_path, index=False)

## Multiple classification for Tara@5


To analyze the stability of the predictions, TARa@5 will be applied on a stratified subset of the dataset. Each example is evaluated five times with the same model, reusing the first prediction and generating four additional executions, allowing the degree of consistency of the outputs to be measured.


In [ ]:
print(f"Starting TARa@5 for dataset: {task} | Texts: {len(df_tara5)}")

output_path = (
    f"/content/drive/MyDrive/tfg/corpusMentalRiskES/resultados/"
    f"{MODEL}_{task}_tara5.csv"
)

remaining_calls = RPD_LIMIT

for run in range(1, 5):
    col_name = f"pred_run{run+1}"

    if col_name in df_tara5.columns:
        continue

    if remaining_calls <= 0:
        print("Daily limit reached. Continue tomorrow.")
        break

    print(f"\n=== Executing {col_name} | Available calls: {remaining_calls} ===")

    df_run = df_tara5[["user_id", "texto"]].copy()

    # IMPORTANT: pass remaining limit
    df_run, used_calls = classify_with_daily_limit(df_run, task, rpd_limit=remaining_calls, run_id=run+1)

    remaining_calls -= used_calls

    df_tara5[col_name] = df_run["pred"]

    df_tara5.to_csv(output_path, index=False)

    print(f"{col_name} completed | calls used: {used_calls} | remaining: {remaining_calls}")

print("\nTARa@5 completed (or in progress if there is a daily limit)")

Iniciando TARa@5 para dataset: anx | Textos: 100

=== Ejecutando pred_run2 | Llamadas disponibles: 240 ===
🆕 Sin checkpoint previo. Empezando desde cero.
📋 Textos pendientes: 100 | Límite de hoy: 240
📅 Días restantes estimados: 1


Progreso de hoy: 100%|██████████| 100/100 [04:48<00:00,  2.89s/it, bloqueados=2, hoy=100]



💾 Checkpoint guardado. Total procesados: 98/100
pred_run2 completado | llamadas usadas: 100 | restantes: 140

=== Ejecutando pred_run3 | Llamadas disponibles: 140 ===
🆕 Sin checkpoint previo. Empezando desde cero.
📋 Textos pendientes: 100 | Límite de hoy: 140
📅 Días restantes estimados: 1


Progreso de hoy: 100%|██████████| 100/100 [04:48<00:00,  2.89s/it, bloqueados=2, hoy=100]



💾 Checkpoint guardado. Total procesados: 98/100
pred_run3 completado | llamadas usadas: 100 | restantes: 40

=== Ejecutando pred_run4 | Llamadas disponibles: 40 ===
🆕 Sin checkpoint previo. Empezando desde cero.
📋 Textos pendientes: 100 | Límite de hoy: 40
📅 Días restantes estimados: 3


Progreso de hoy:  35%|███▌      | 14/40 [00:34<01:03,  2.46s/it, bloqueados=0, hoy=14]

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_requests_per_model_per_day, limit: 250, model: gemini-3.1-pro\nPlease retry in 3h45m32.695956039s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_requests_per_model_per_day', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel', 'quotaDimensions': {'model': 'gemini-3.1-pro', 'location': 'global'}, 'quota

In [12]:
checkpoint_manual = checkpoint_dir + f"tara5_{MODEL.replace('/', '_')}_{task}_run{run+1}.csv"
df_run.to_csv(checkpoint_manual, index=False)
print(f"💾 Manual safety save: {checkpoint_manual}")

💾 Guardado manual de seguridad: /content/drive/MyDrive/tfg/corpusMentalRiskES/checkpoints/tara5_gemini-3.1-pro-preview_anx_run4.csv
